# Thermal Network Excel Import

This notebook demonstrates how to import thermal network configurations from Excel files
and convert them to YAML configuration files for simulation.

## Overview

The Excel-based workflow simplifies creating complex thermal network scenarios by:

1. **Single Excel file input** with multiple sheets for different components
2. **Automatic return pipe generation** from supply pipes
3. **Automatic node extraction** from connections
4. **Built-in validation** before YAML generation
5. **Support for Brownfield and Greenfield** scenarios

## Excel Structure

The Excel file should contain 6 sheets:

- **Netzwerk**: Network-level parameters (temperatures, pressure, etc.)
- **Erzeuger**: Heat producers (boilers, heat pumps, CHP, etc.)
- **Speicher**: Thermal storage units
- **Rohre**: Pipe definitions (ONLY supply pipes - return pipes auto-generated!)
- **Verbraucher**: Heat consumers with demand profiles
- **Zeitreihen**: All timeseries data (8760 rows for full year)

## Brownfield vs Greenfield

Each component can be marked as:
- **Existing (Brownfield)**: `Bestand? = ja` → Fixed geometry, CAPEX = 0
- **Investment (Greenfield)**: `Investition? = ja` → Optimize geometry, calculate CAPEX

This allows modeling of:
- Pure Greenfield: All components are new
- Pure Brownfield: All components exist, optimize operation only
- **Mixed scenarios**: Extend existing network with new components

## Step 1: Create Excel Template

First, create an Excel template with the correct structure:

In [ ]:
# Create template (requires openpyxl for multi-sheet, otherwise creates simple template)
!python ../scripts/create_thermal_network_template.py --output ../templates/my_network.xlsx

## Step 2: Fill in Your Data

Open the template in Excel and fill in your network data:

1. **Netzwerk sheet**: Set network-level parameters
2. **Erzeuger sheet**: Define heat producers (existing and/or investment)
3. **Speicher sheet**: Define storage units (if any)
4. **Rohre sheet**: Define ONLY supply pipes (return pipes generated automatically)
5. **Verbraucher sheet**: Define consumers with demand profiles
6. **Zeitreihen sheet**: Add all timeseries (8760 hours)

Save the file when done.

## Step 3: Parse Excel to YAML

Use the `ThermalNetworkExcelParser` to convert Excel to YAML configuration:

In [ ]:
from energis.utils.thermal_network_excel_parser import ThermalNetworkExcelParser

# Path to your filled-in Excel file
excel_path = "../data/my_network.xlsx"

# Create parser
parser = ThermalNetworkExcelParser(excel_path)

# Show summary
print(parser.get_summary())

## Step 4: Validate Configuration

Before saving, validate that the configuration is correct:

In [ ]:
# Validate configuration
errors = parser.validate()

if errors:
    print("❌ Validation errors found:")
    for i, error in enumerate(errors, 1):
        print(f"  {i}. {error}")
else:
    print("✓ Configuration is valid!")

## Step 5: Save YAML Configuration

If validation passes, save the configuration to YAML:

In [ ]:
if not errors:
    # Save to configs/scenarios/
    output_path = "../configs/scenarios/my_network.scenario.yaml"
    parser.save_yaml(output_path)
    
    print(f"\n✓ Configuration saved to: {output_path}")
    print(f"✓ Timeseries saved to: {output_path.replace('.scenario.yaml', '_timeseries.csv')}")
else:
    print("\n❌ Fix validation errors before saving")

## Step 6: Inspect Generated Configuration

You can inspect the generated configuration:

In [ ]:
import yaml

# Display thermal network config
print("Thermal Network Configuration:")
print(yaml.dump(parser.config['thermal_network'], default_flow_style=False))

print("\nProducers:")
for producer in parser.config['producers']:
    print(f"  - {producer['id']}: {producer['type']} (existing={producer.get('existing')}, invest={producer.get('invest')})")

print("\nPipes (including auto-generated return pipes):")
for pipe in parser.config['pipes']:
    print(f"  - {pipe['id']}: {pipe['from_node']} → {pipe['to_node']} ({pipe['length_m']}m)")

## Example: Brownfield Extension Scenario

Here's a complete example of extending an existing district heating network with a heat pump and storage:

In [ ]:
# This would be filled in your Excel file:

example_config = """
ERZEUGER SHEET:
ID          Typ         Bestand?  Investition?  Q_nom_MW  Q_options_MW  CAPEX_€_kW  ...
Kessel_1    boiler      ja        nein          5.0                     300
WP_1        heat_pump   nein      ja                      1.0,2.0,3.0   800

SPEICHER SHEET:
ID          Typ             Bestand?  Investition?  Kapazitaet_MWh  Kapazitaet_options_MWh  ...
Speicher_1  hot_water_tank  nein      ja                            5,10,20

ROHRE SHEET (nur Vorlauf!):
ID   Von_Knoten  Zu_Knoten  Laenge_m  Bestand?  Investition?  DN_fix  DN_options
P1   N1          N2         500       ja        nein          DN150
P2   N2          N3         300       ja        nein          DN100
P3   N3          N4         200       nein      ja                    DN100,DN150,DN200

Result:
- Existing boiler (5 MW) at fixed capacity
- NEW heat pump with 3 size options (1, 2, or 3 MW)
- NEW storage with 3 size options (5, 10, or 20 MWh)
- Existing pipes P1 and P2 with fixed DN
- NEW pipe P3 with DN optimization
- Return pipes P1_return, P2_return, P3_return auto-generated

Optimizer will:
1. Choose optimal heat pump size (1, 2, or 3 MW)
2. Choose optimal storage size (5, 10, or 20 MWh)
3. Choose optimal DN for P3 (DN100, DN150, or DN200)
4. Optimize operation of all components
5. Calculate total costs (OPEX + CAPEX for new components)
"""

print(example_config)

## Troubleshooting

### Common Validation Errors

1. **"Cannot have both existing=True and invest=True"**
   - Fix: Each component must be EITHER existing OR investment, not both

2. **"invest=True requires Q_options/capacity_options/DN_options"**
   - Fix: Investment components need a list of options for the optimizer

3. **"existing=True requires Q_nom/capacity_MWh/DN_fixed"**
   - Fix: Existing components need fixed parameters

4. **"from_node 'X' not found"**
   - Fix: Check spelling of node names, they are auto-extracted from connections

5. **"demand_profile 'X' not found in timeseries"**
   - Fix: Make sure column names in Zeitreihen sheet match profile names in Verbraucher

### Performance Tips

- Start with small test cases (24-168 hours) before running full year
- Limit investment options to 3-5 choices per component
- Use existing=True for components where you don't want optimization

### Multi-Sheet Excel Support

If `openpyxl` is not installed, the template generator creates a simple single-sheet template.
Install openpyxl for full multi-sheet support:

```bash
pip install openpyxl
```

## Next Steps

After creating the YAML configuration:

1. Use the configuration in simulation runners
2. Visualize results with analysis notebooks
3. Compare different scenarios (e.g., with/without storage)
4. Perform sensitivity analysis on key parameters

See other notebooks for simulation and analysis workflows.